<a href="https://colab.research.google.com/github/innocenti2010/Data-Scientist-Portfolio/blob/main/Python-Projects/%F0%9F%93%A6%20Cross-Selling-Insurance-ML/Cross_Selling_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cross-Selling Assicurativo**

**AssurePredict** è una compagnia di assicurazioni leader nel settore, specializzata nell'offrire soluzioni innovative per la gestione del rischio. Questo progetto mira a creare un modello predittivo in grado di individuare potenziali opportunità di **cross-selling** per clienti esistenti, identificando quelli che potrebbero essere interessati ad acquistare una polizza aggiuntiva per il loro veicolo.

## **OBIETTIVO DEL PROGETTO**

L'obiettivo è sviluppare un **modello di machine learning** che preveda se i clienti, che attualmente hanno un'assicurazione sanitaria, potrebbero essere interessati a sottoscrivere una polizza assicurativa per il loro veicolo. Il modello aiuterà AssurePredict a migliorare l'efficacia delle proprie strategie di cross-selling e ad aumentare la penetrazione nel mercato.

**Valore aggiunto per Assure Predict:**

- **Aumento del tasso di conversione** nelle vendite di polizze auto.
- **Ottimizzazione delle campagne di marketing**, indirizzando le offerte a clienti più propensi ad acquistare.
- **Riduzione dei costi** legati a campagne di marketing inefficaci, grazie alla targettizzazione precisa.


**Dataset**

Il dataset (che è scaricabile da qui:  https://proai-datasets.s3.eu-west-3.amazonaws.com/insurance_cross_sell.csv) contiene informazioni dettagliate sui clienti e sul loro comportamento assicurativo. Le caratteristiche principali del dataset sono:
- **id**: identificativo univoco del cliente.
- Gender: sesso del cliente.
- **Age**: età del cliente.
- **Driving_License**: 1 se il cliente possiede la patente di guida, 0 altrimenti.
- **Region_Code**: codice univoco della regione di residenza del cliente.
- **Previously_Insured**: 1 se il cliente ha già un veicolo assicurato, 0 altrimenti.
- **Vehicle_Age**: età del veicolo del cliente.
- **Vehicle_Damage**: 1 se il cliente ha avuto incidenti o danni al veicolo in passato, 0 altrimenti.
- **Annual_Premium**: importo annuale del premio assicurativo pagato dal cliente.
- **PolicySalesChannel**: canale utilizzato per la vendita della polizza (es. email, telefono, di persona).
- **Vintage**: giorni da cui il cliente è assicurato con AssurePredict.
- **Response**: 1 se il cliente ha accettato la proposta di cross-sell, 0 altrimenti.


# **1. ESPLORAZIONE DEL DATASET**

L'esplorazione preliminare del dataset permetterà di comprendere meglio la distribuzione delle caratteristiche e delle variabili target. In particolare, si analizzeranno:
- La distribuzione della variabile "Response", per identificare eventuali sbilanciamenti tra clienti che accettano o rifiutano l'offerta di cross-sell.
- Le relazioni tra variabili chiave come Annual Premium, Vehicle Age, Previously Insured, e la risposta del cliente.<br>

**Valore aggiunto**: Un'accurata esplorazione dei dati permette di identificare pattern nascosti e punti critici che influenzeranno il successo del modello predittivo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## **1.1 Caricamento Dati e prima analisi del dataset**

---

In questa prima fase, carico il dataset e verifico la correttezza dei campi del file. Verifico la presenza di valori mancanti e se se le variabili si trovano in un formato utilizzabile. Tutto ciò per assicurarmi che i dati siano pronti per l'analisi esplorativa.

In [ ]:
df=pd.read_csv("/content/insurance_cross_sell.csv", index_col=0)
df.head()

In [ ]:
print("DIMENSIONI DATASET Cross-Selling")
print(f"ROWS={df.shape[0]}")
print(f"COLUMNS={df.shape[1]}")

In [ ]:
df.info()

In [ ]:
df.describe()

Dalla prima analisi esplorativa, noto che le variabili qualitative sono: `Gender`, `Vehicle_Age`, `Vehicle_Damage`. <br>
Queste variabili devono essere trasformate in numeriche per poter essere utilizzate in modelli di machine learning.                         

## **1.2 Trasformazione Dati Qualitativi in Quantitativi**

In [ ]:
Gender=df['Gender'].values
np.unique(Gender)


In [ ]:
vehicle_age = df['Vehicle_Age'].values
np.unique(vehicle_age)

In [ ]:
Vehicle_Damage= df['Vehicle_Damage'].values
np.unique(Vehicle_Damage)

Prima di procedere, rimuovo la colonna indice `id` dal dataframe, perchè non contiene informazioni utili per la predizione.

In [ ]:
df_copy=df.copy()
df_copy = df_copy.reset_index(drop=True)
df_copy.head()

Applico una mappatura delle variabili qualitative.

In [ ]:
gender_mapping={'Female':0, 'Male':1}
age_mapping= {'1-2 Year':1, '< 1 Year':2, '> 2 Years':3}
damage_mapping={'No':0, 'Yes':1}

In [ ]:
df_copy['Gender']=df_copy['Gender'].map(gender_mapping)
df_copy['Vehicle_Age']= df_copy['Vehicle_Age'].map(age_mapping)
df_copy['Vehicle_Damage']= df_copy['Vehicle_Damage'].map(damage_mapping)
df_copy.head()


Statistiche Genarali

In [ ]:
df_copy.count()

In [ ]:
df_copy.isna().sum()

In [ ]:
df_copy[df_copy["Vintage"].isna()]
df_copy[df_copy["Response"].isna()]

In [ ]:
df_copy.info()

In [ ]:
df_copy['Age'] = pd.to_numeric(df_copy['Age'], errors='coerce')
df_copy['Age'] = df_copy['Age'].fillna(df_copy['Age'].median())


## **1.3 Verifica Correlazioni tra variabili**

---
Per avere una visione generale delle relazioni tra varibili, calcolo la matrice di correlazione. <br>
Cò mi permette di verificare la relazione tra la variabile target `Response` e le altre variabili, evidenziando le varibili che sono maggiormente correlate, con lo scopo di sfruttarle per generare il modello predittivo risultante.


In [ ]:
plt.figure(figsize=(14,10),dpi=100)
heatmap=sns.heatmap(df_copy.corr(),
                    cbar= True,
                    square= True,
                    xticklabels=df_copy.columns,
                    yticklabels=df_copy.columns,
                    annot= True,
                    annot_kws={'size':12})

plt.show()

Dalla matrice si osserva che le variabili più correlate con Response sono:
- vehicle_damage (correlazione positiva)
- previously_insured (correlazione negativa)
- policy_sale_channel (correlazione debole)
- Age (correlazione debole)
- vehicle_age (correlazione debole)


## **1.4 Analisi della variabile target: Response**

In [ ]:
df_copy['Response'].value_counts()
df['Response'].value_counts(normalize=True)


La variabile Response risulta sbilanciata: la maggior parte dei clienti non accetta l'offerta di cross-sell. Quindi in fase di predizione del modello, bisogna gestire lo sbilanciamento, perchè il modello potrebbe imparare a predire molto spesso il valore 0.

## **1.5 Visualizzazione delle Distribuzioni delle variabili**

---

Per analizzare meglio le relazioni tr Response e alcune variabili chiave, utilizzo vari grafici.

In [ ]:
plt.figure(figsize=(14,10), dpi=100)
plt.subplot(2,2,1)
plt.title('Age vs Response')
sns.boxplot(x='Response',y='Age', data=df_copy, palette='viridis', hue='Response', legend=False)
plt.subplot(2,2,2)
plt.title('Policy Sales Channel vs Response')
sns.boxplot(x='Response',y='Policy_Sales_Channel', data=df_copy, palette='viridis', hue='Response', legend=False)
plt.subplot(2,2,3)
plt.title('Response vs Vehicle Damage')
sns.boxplot(x='Response',y='Vehicle_Damage', data=df_copy, palette='viridis', hue='Response', legend=False)
plt.ylim(-0.5,1.5)
plt.subplot(2,2,4)
plt.title('Response vs Previously insured')
sns.boxplot(x='Response',y='Previously_Insured', data=df_copy, palette='viridis', hue='Response', legend=False)
plt.ylim(-0.5,1.5)

I boxplot non mostrano in modo chiaro la relazione tra Response e le 2 variabili binarie `vehicle_damage` e `previously_insured`. Quindi proviamo a visulizzarli tramite istogrammi.

In [ ]:
plt.figure(figsize=(10,10),dpi=100)
plt.subplot(2,2,1)
sns.histplot(df_copy[df_copy['Response'] == 0]['Vehicle_Damage'], color='blue', label='Response = 0', kde=False, bins=4)
sns.histplot(df_copy[df_copy['Response'] == 1]['Vehicle_Damage'], color='red', label='Response = 1', kde=False, bins=4)
plt.xlabel('Vehicle Damage')
plt.ylabel('Frequency')
plt.title("Vehicle Damage vs Response")
plt.xlim(-0.5, 1.5)
plt.xticks([0, 1], ['No', 'Yes'])
plt.legend()
plt.grid(True)
plt.subplot(2,2,2)
sns.histplot(df_copy[df_copy['Response'] == 0]['Previously_Insured'], color='blue', label='Response = 0', kde=False, bins=4)
sns.histplot(df_copy[df_copy['Response'] == 1]['Previously_Insured'], color='red', label='Response = 1', kde=False, bins=4)
plt.xlabel('Vehicle Damage')
plt.ylabel('Frequency')
plt.title("Previously Insured vs Response")
plt.xlim(-0.5, 1.5)
plt.xticks([0, 1], ['No', 'Yes'])
plt.legend()
plt.grid(True)
plt.show()

**ANALISI GRAFICI**

---

**1. Age** <br>
I clienti che accettano l'offerta tendono ad avere una fascia d'età che mediamente è compresa tra i 40-50 anni. Questo suggerisce che le fasce più giovani siano meno propensi ad accettare tale offerta.
<br>

**2. Policy Sales Channel** <br>
Non emergono differenze significative tra i 2 gruppi.
<br>

**3. Vehicle Damage** <br>
I clienti che hanno avuto danni al veicolo sembrano più propensi ad accettare l’offerta. Probabilmente percepiscono un rischio maggiore.
<br>

**4. Previously Insured** <br>
I clienti che hanno già un veicolo assicurato tendono a non essere interessati a una nuova polizza. Infatti se sono già coperti è meno probabile che accettino una nuova polizza.

## **1.6 CONCLUSIONE EDA**

L'esplorazione del dataset mostra che la variabile Response è sbilanciata poichè la maggioranza dei clienti non accetta l'offerta della polizza. Alcune variabili come `Vehicle_Damage` e `Previously_Insured` mostrano una chiara relazione con la risposta del cliente e quindi saranno utili per la costruzione del modello e nella gestione dello sbilanciamento.

# **2. SBILANCIAMENTO DELLE CLASSI**

La variabile target `Response` potrebbe essere sbilanciata, con molti più clienti che rifiutano l'offerta rispetto a quelli che la accettano. Per affrontare questo problema, verranno utilizzate tecniche di:
- **Class Weights**: penalizzazione della classe più frequente nel modello.
- **Oversampling o Undersampling**: creazione di un dataset più bilanciato per migliorare la capacità del modello di generalizzare.

**Valore aggiunto**: Gestire correttamente lo sbilanciamento delle classi è cruciale per evitare modelli che abbiano un alto tasso di falsi negativi, migliorando così la precisione del cross-sell.

---

Prima però è necessario preparare correttamente i dati.


## **2.1 Preparazione del dataset**

---
Divido il dataset in variabili indipendenti (X) e variabile target (y)


In [ ]:
X=df_copy.drop(['Response'], axis=1).values
y=df_copy['Response'].values

Per semplificare la fase di pre-processing, creo una versione con le varibili più correlate.

In [ ]:
X_short = df_copy[["Previously_Insured", "Vehicle_Damage", "Policy_Sales_Channel", "Age"]].values

Stampo i valori minimi e massimi per verificare le scale delle variabili.

In [ ]:
print("Xmin - Xmax:",X.min(), X.max())
print("X_short min - Xmax:",X_short.min(), X_short.max())
print("ymin - ymax:",y.min(), y.max())

## **2.2 Train-tests split**

---
Divido i dati in training e test set, mantenendo la proporzione delle classi con stratify=y


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test,y_train, y_test = train_test_split(X,y, stratify=y)
X_short_train, X_short_test, y_short_train, y_short_test= train_test_split(X_short, y, stratify=y)

## **2.3 Standardizzazione**

---
Applico la standardizzazione sul training set, e poi trasformo il test set con i medesimi parametri.


In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
ss= StandardScaler()
X_train= ss.fit_transform(X_train)
X_test= ss.transform(X_test)
X_short_train= ss.fit_transform(X_short_train)
X_short_test= ss.transform(X_short_test)

Generalmente i modelli di ml lavorano meglio quando le variabili sono sulla stessa scala . Inoltre tecniche come oversampling e undersampling si basano su distanze di punti, quindi è importante una loro precedente standardizzazione.

## **2.4 Oversampling**

---
L'oversampling consiste nell'aumentare la classe minoritaria duplicando o generando nuovi esempi.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

In [ ]:
oversample= RandomOverSampler(sampling_strategy='minority')
X_over, y_over=oversample.fit_resample(X_train,y_train)
X_short_over, y_short_over= oversample.fit_resample(X_short_train,y_short_train)

Vado ad applicare solo al training set, perchè nel caso in cui aggiungessi dati sintetici del test avrei la valutazione del modello non realistica.

In [ ]:
#Confronto tra distribuzione originale e dopo Oversampling
print("Originale:", Counter(y))
print("Oversampling:", Counter(y_short_over))

## **2.5 UnderSampling**

---
L'undersampling riduce la classe maggioritaria eliminando alcuni esempi.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
under_sampling= RandomUnderSampler(sampling_strategy='majority')
X_under,y_under= under_sampling.fit_resample(X_train,y_train)
X_short_under, y_short_under= under_sampling.fit_resample(X_short_train,y_short_train)

In [ ]:
#Confronto tra distribuzione originale e dopo Undersampling
print("Originale:", Counter(y))
print("Oversampling:", Counter(y_short_under))

## **2.6 Conclusione Step 2**

Dato che la variabile Response risultava fortemente sbilanciata, ho appliicato le tecniche di oversampling e undersampling sul training set. <br>
Queste strategie permettono di migliorare la capacità del modello di riconoscere i clienti realmente interessati al cross_sell. La class-weight lo andrò ad applicare nel modello (Logistic Regression).

# **3. COSTRUZIONE DEL MODELLO PREDITTIVO**

Utilizzando algoritmi di machine learning, verrà costruito un modello che predice la probabilità che un cliente risponda positivamente all'offerta di cross-sell. <br>

**Valore aggiunto**: Il modello predittivo permetterà a **AssurePredict** di identificare con precisione i clienti più propensi a sottoscrivere una polizza aggiuntiva, migliorando così il ritorno sull'investimento delle campagne di marketing.

Per creare il modello, utilizzo la **Regressione Logistica**, affiancandolo e verificandolo per le tecniche di bilanciamento:
- **class_weight= 'balanced'**
- **oversampling**
- **undersampling**

In questo modo posso confrontare le diverse strategie e capire quale funziona meglio.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

## **3.1 Definizione Dataset INPUT**

Prima di addestrare i modelli, riepilogo i dataset disponibili. <br>
Questi Dataset li andrò ad utilizzare per addestrare e confrontare i modelli.

---

▶ **STANDARDIZZATI**
- X,y = Dataset completo e standardizzato
- X_short, y_short = Dataset con variabili più correlate standardizzate

**NB**: Entrambi hanno versione di Training e Testing

▶ **STANDARDIZZATI & CON OVERSAMPLING**
- X_over, y_over = Dataset con oversampling e standardizzato
- X_short_over, y_short_over = Dataset con variabili più correlate  standardizzate

**NB**: Entrambi hanno versione di Training e Testing

▶ **STANDARDIZZATI & CON UNDERSAMPLING**
- X_under, y_under = Dataset con undersampling e standardizzato
- X_short_under, y_short_under = Dataset con variabili più correlate standardizzate

**NB**: Entrambi hanno versione di Training e Testing


In [ ]:
full_train = (X_train, y_train)
full_test = (X_test, y_test)
print("full dataset train: ", full_train[0].shape, full_train[1].shape)
print("full dataset test: ", full_test[0].shape, full_test[1].shape)
print("-----------")

short_train = (X_short_train, y_short_train)
short_test = (X_short_test, y_short_test)
print("Short dataset train: ", X_short_train[0].shape, y_short_train[1].shape)
print("Short dataset test: ", X_short_test[0].shape, y_short_test[0].shape)
print("-----------")

oversampled_db_train = (X_over, y_over)
oversampled_db_test = (X_test, y_test)
print("oversampled dataset train: ", oversampled_db_train[0].shape, oversampled_db_train[1].shape)
print("oversampled dataset test: ", oversampled_db_test[0].shape, oversampled_db_test[1].shape)
print("-----------")

oversampled_db_short_train = (X_short_over, y_short_over)
oversampled_db_short_test = (X_short_test, y_short_test)
print("oversampled dataset short train: ", oversampled_db_short_train[0].shape, oversampled_db_short_train[1].shape)
print("oversampled dataset short test: ", oversampled_db_short_test[0].shape, oversampled_db_short_test[1].shape)
print("-----------")

undersampled_db_train= (X_under, y_under)
undersampled_db_test = (X_test, y_test)
print("undersampled dataset short train: ", undersampled_db_train[0].shape, undersampled_db_train[1].shape)
print("undersampled dataset short test: ", undersampled_db_test[0].shape, undersampled_db_test[1].shape)
print("-----------")

undersampled_db_short_train = (X_short_under, y_short_under)
undersampled_db_short_test = (X_short_test, y_short_test)
print("undersampled dataset short train: ", undersampled_db_short_train[0].shape, undersampled_db_short_train[1].shape)
print("undersampled dataset short test: ", undersampled_db_short_test[0].shape, undersampled_db_short_test[1].shape)
print("-----------")

**FUNZIONE DI VALUTAZIONE**

Definisco Metriche di Valutazione in un'unica funzione:

- **Classification Report** : Sintesi delle principali metriche di valutazione di un modello di classificazione (Specify, Accuracy, Recall, Precision).
- **Matrice di Confusione** : Permette di sapere non solo quanti errori ha commesso il modello ma anche quali.
- **ROC Curve** : Permette di verificare le performance di un modello di classificazione per diversi valori di threshold.

In [ ]:
def evaluate_model(model, data,
                   labels=('Negative', 'Positive'),
                   show_precision_recall=True):

    # dati
    X, y = data

    # prediction
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    # metriche
    acc = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)

    print("========== CLASSIFICATION REPORT ==========")
    print(classification_report(y, y_pred))

    print(f"Accuracy : {acc:.4f}")
    print(f"AUC      : {auc:.4f}")

    # ======================
    # CONFUSION MATRIX
    # ======================

    cm = confusion_matrix(y, y_pred)

    df_cm = pd.DataFrame(
        cm,
        index=labels,
        columns=[f'Predicted {labels[0]}',
                 f'Predicted {labels[1]}']
    )

    plt.figure(figsize=(6,5))
    sns.heatmap(df_cm, annot=True, fmt='g', cmap='Blues')

    if show_precision_recall:
        precision = cm[1][1] / (cm[1][1] + cm[0][1])
        recall = cm[1][1] / (cm[1][1] + cm[1][0])

        plt.text(0, -0.3,
                 f'Precision: {precision:.3f}',
                 fontsize=11)

        plt.text(1, -0.3,
                 f'Recall: {recall:.3f}',
                 fontsize=11)

    plt.title("Confusion Matrix")
    plt.show()

    # ======================
    # ROC CURVE
    # ======================

    fpr, tpr, thresholds = roc_curve(y, y_proba)

    plt.figure(figsize=(6,5))

    plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    plt.plot([0,1], [0,1], linestyle='--')

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()

    plt.show()

## **3.2 LOGISTIC REGRESSION**

---

La prima versione del modello utilizza il dataset standardizzato e il parametro `class_weight= 'balanced'`, che assegna in automatico un peso maggiore alla classe maggioritaria.

In [ ]:
lr=LogisticRegression(class_weight='balanced')
lr.fit(full_train[0],full_train[1])


### 3.2.1 VALUTAZIONE - Logistic Regression

In [ ]:
print("Full Dataset - TRAIN REPORT")
lr.fit(full_train[0], full_train[1])
evaluate_model(lr, (full_train[0],full_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Full Dataset - TEST REPORT")
#lr.fit(full_test[0], full_test[1])
evaluate_model(lr, (full_test[0],full_test[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**📊 Risultati train vs test set**
- Accuracy ≈ 0.6387  | 0.6410
- AUC ≈      0.8197  | 0.8210
- Recall classe 1 molto alto (≈ 0.98)
- Precision classe 1 bassa (≈ 0.25)

**COMMENTO**: <br>
Le metriche su train e testing set sono molto simili: il modello è stabile e non mostra segni evidenti di overfitting.

La matrice di confusione mostra che il modello intercetta quasi tutti i positivi (alto recall), ma con molti falsi positivi (bassa precision).

In [ ]:
print("Short Dataset - TRAIN REPORT")
lr.fit(short_train[0], short_train[1])
evaluate_model(lr, (short_train[0],short_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Short Dataset - TEST REPORT")
lr.fit(short_train[0], short_train[1])
evaluate_model(lr, (short_train[0],short_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**📊 Risultati train vs test set**
- Accuracy ≈ 0.6382
- AUC ≈      0.8177
- Recall classe 1 molto alto (≈ 0.98)
- Precision classe 1 bassa (≈ 0.25)

**COMMENTO**: <br>
Prendendo solo le variabili maggiormente correlate alla variabile target, AUC diminuisce fino al 0.818, ma non c'è grande differenza tra train e test. Anche qui abbiamo alta recall e bassa Precision come nel caso in cui vengono prese tutte le variabili.

## **3.3 LOGISTIC REGRESSION - Oversampling**

In [ ]:
lr_over = LogisticRegression(class_weight='balanced')
lr_over.fit(oversampled_db_train[0], oversampled_db_train[1])


### 3.3.1 VALUTAZIONE - Logistic Regression - Oversampling

In [ ]:
print("Oversampled FULL Dataset - TRAIN REPORT")
evaluate_model(lr_over, (oversampled_db_train[0],oversampled_db_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Oversampled FULL Dataset - TEST REPORT")
evaluate_model(lr_over, (oversampled_db_test[0],oversampled_db_test[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**📊 Risultati train vs test set**
- Accuracy : 0.7838 | 0.6410
- AUC :      0.8194 | 0.8210
- Recall classe 1 molto alto (≈ 0.98)
- Precision classe 1 bassa (≈ 0.251)

**COMMENTO**: <br>
L'oversampling migliora la precisione in fase di training, mantenendo un recall molto alto, ma in fase di testing le prestazioni sono identiche al modello con class-weight. <br>
Quindi migliora il training ma non porta benefici sul testing.


In [ ]:
lr_over_short = LogisticRegression(class_weight='balanced') #
lr_over_short.fit(oversampled_db_short_train[0], oversampled_db_short_train[1])

In [ ]:
print("Oversampled SHORT Dataset - TRAIN REPORT")
evaluate_model(lr_over_short, (oversampled_db_short_train[0],oversampled_db_short_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Oversampled SHORT Dataset - TEST REPORT")
evaluate_model(lr_over_short, (oversampled_db_short_test[0],oversampled_db_short_test[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**📊 Risultati train vs test set**
- Accuracy : 0.7839 | 0.6378
- AUC      : 0.8178 | 0.8178
- Recall classe 1 molto alto (≈ 0.98)
- Precision classe 1 bassa (≈ 0.25)

## **3.4 LOGISTIC REGRESSION - Undersampling**

In [ ]:
lr_under = LogisticRegression(class_weight='balanced')
lr_under.fit(undersampled_db_train[0], undersampled_db_train[1])


### 3.4.1 VALUTAZIONE - Logistic Regression - Undersampling

In [ ]:
print("UnderSampled FULL Dataset - TRAIN REPORT")
evaluate_model(lr_under, (undersampled_db_train[0],undersampled_db_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Undersampled FULL Dataset - TEST REPORT")
evaluate_model(lr_under, (undersampled_db_test[0],undersampled_db_test[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**📊 Risultati train vs test set**
- Accuracy : 0.7845 | 0.6411
- AUC      : 0.8218 | 0.8208
- Recall classe 1 molto alto (≈ 0.98)
- Precision classe 1 bassa (≈ 0.252)

**COMMENTO**: <br>
L'undersampling riduce la classe maggioritaria, bilanciando il dataset ma eliminando molti esempi reali. <br>
Qui anche se il training mostra valori più alti, le prestazioni sul test sono simili agli altri modelli.

In [ ]:
lr_under_short = LogisticRegression(class_weight='balanced')
lr_under_short.fit(undersampled_db_short_train[0], undersampled_db_short_train[1])

In [ ]:
print("Undersampled SHORT Dataset - TRAIN REPORT")
evaluate_model(lr_under_short, (undersampled_db_short_train[0],undersampled_db_short_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

In [ ]:
print("Undersampled SHORT Dataset - TEST REPORT")
evaluate_model(lr_under_short, (undersampled_db_short_train[0],undersampled_db_short_train[1]),
                   labels=('NO', 'YES'),
                   show_precision_recall=True )

**COMMENTO**: <br>
Anche con lo short dataset, l'undersampling ha le stesse performance del dataset completo. Alte performance in training ma ridotte in fase di Testing.

## **3.5 SUMMARY METRICHE**

| MODELLO | MODE | Accuracy | AUC | Precision | Recall |
|---|---|---|---|---|---|
| Logistic Regression | TRAIN | 0.6387 | 0.8197 | 0.25 | 0.98 |
| Logistic Regression | TEST | 0.6410 | 0.8210 | 0.25 | 0.98 |
| Logistic Regression - Oversampling | TRAIN | 0.7838 | 0.8194 | 0.70 | 0.98 |
| Logistic Regression - Oversampling | TEST | 0.6410 | 0.8217 | 0.251 | 0.98 |
| Logistic Regression - Undersampling | TRAIN | 0.7845 | 0.8218 | 0.71 | 0.98 |
| Logistic Regression - Undersampling | TEST | 0.6411 | 0.8208 | 0.25 | 0.98 |

**Osservazioni**

---

- Le metriche sul test set sono quasi identiche per tutti i modelli.
- L'oversampling e l'undersampling migliorano artificialmente le metriche sul training ma non portano vantaggi sul test.
- La matrice di confusione mostra in tutti i casi una recall molto alta per la classe 1, a fronte di una precisione + bassa per la LR, coerente con lo sbilanciamento del dataset.

# **4. SCELTA DEL MODELLO**

Dopo aver confrontato i vari modelli, si può osservare che le performance sul test sono molto simili: quindi il modello finale scelto è la **Logistic Regression con class_weight = 'balanced'**. <br>
Questo modello è semplice, stabile, non richiede la modifica artificiale del dataset e mantiene un'AUC elevata (=0.82), con una recall alta sulla classe 1. <br>
Dal punto di vista del business, questo permette ad AssurePredict d'identificare la maggior parte dei clienti interessati al cross-sell, concentrando le campagne marketing su profili + promettenti.
